In [ ]:
# ============================================================
# 02_train_llama31_8b_qlora.py
# 목적:
#   Llama 3.1 8B 계열 모델을 QLoRA 방식으로 파인튜닝
#
# 입력:
#   train.jsonl
#   validation.jsonl
#
# 출력:
#   LoRA adapter
#   merged model
# ============================================================

# ------------------------------------------------------------
# 0. 설치
# ------------------------------------------------------------
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install --no-deps trl peft accelerate bitsandbytes datasets transformers sentencepiece protobuf

import os
import json
import torch
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")


# ------------------------------------------------------------
# 1. 경로 설정
# ------------------------------------------------------------
BASE_DIR = Path("/content/drive/MyDrive/Army_Finetune")

TRAIN_JSONL = BASE_DIR / "02_jsonl" / "train.jsonl"
VAL_JSONL = BASE_DIR / "02_jsonl" / "validation.jsonl"

OUTPUT_DIR = BASE_DIR / "03_outputs"
LORA_DIR = OUTPUT_DIR / "army_lora_adapter"
MERGED_DIR = OUTPUT_DIR / "army_merged_model"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. 모델 설정
# ------------------------------------------------------------
# 주의:
# gated model이면 Hugging Face 로그인이 필요할 수 있다.
# Meta-Llama-3.1-8B-Instruct 접근 권한이 없다면 아래 대체 모델을 사용.
#
# 대체 후보:
#   "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
#   "unsloth/Llama-3.1-8B-Instruct-bnb-4bit"

MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = True


# ------------------------------------------------------------
# 3. 모델 로드
# ------------------------------------------------------------
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)


# ------------------------------------------------------------
# 4. LoRA 설정
# ------------------------------------------------------------
# r:
#   LoRA의 표현력. 16 또는 32 권장.
#
# target_modules:
#   Llama 계열에서 많이 쓰는 projection layer.
#
# lora_alpha:
#   보통 r의 2배 정도.
#
# lora_dropout:
#   데이터가 적을 때 0.05도 가능하지만, Unsloth는 0을 권장하는 경우가 많음.

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)


# ------------------------------------------------------------
# 5. 데이터셋 로드
# ------------------------------------------------------------
from datasets import load_dataset

data_files = {
    "train": str(TRAIN_JSONL),
    "validation": str(VAL_JSONL),
}

dataset = load_dataset("json", data_files=data_files)

print(dataset)


# ------------------------------------------------------------
# 6. Chat template 적용
# ------------------------------------------------------------
# JSONL 구조:
# {
#   "messages": [
#     {"role": "system", "content": "..."},
#     {"role": "user", "content": "..."},
#     {"role": "assistant", "content": "..."}
#   ]
# }

def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}


dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

print(dataset["train"][0]["text"][:2000])


# ------------------------------------------------------------
# 7. 학습 설정
# ------------------------------------------------------------
from trl import SFTTrainer
from transformers import TrainingArguments

# 데이터 100개 수준의 초소형 실험 기준:
#   num_train_epochs = 3~5
# 데이터가 500~1000개로 늘면:
#   num_train_epochs = 2~3
#
# 과적합이 의심되면:
#   epoch 줄이기
#   learning_rate 낮추기
#   데이터 늘리기

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"] if len(dataset["validation"]) > 0 else None,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir=str(OUTPUT_DIR / "training_checkpoints"),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        num_train_epochs=4,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
        save_strategy="epoch",
        eval_strategy="epoch" if len(dataset["validation"]) > 0 else "no",
    ),
)


# ------------------------------------------------------------
# 8. 학습 실행
# ------------------------------------------------------------
trainer_stats = trainer.train()

print(trainer_stats)


# ------------------------------------------------------------
# 9. LoRA 어댑터 저장
# ------------------------------------------------------------
model.save_pretrained(str(LORA_DIR))
tokenizer.save_pretrained(str(LORA_DIR))

print("LoRA adapter 저장 완료:", LORA_DIR)


# ------------------------------------------------------------
# 10. 병합 모델 저장
# ------------------------------------------------------------
# Colab 메모리가 부족하면 이 단계에서 터질 수 있다.
# 그 경우 LoRA adapter만 저장하고, 로컬 PC나 더 큰 GPU에서 병합한다.

model.save_pretrained_merged(
    str(MERGED_DIR),
    tokenizer,
    save_method="merged_16bit",
)

print("Merged model 저장 완료:", MERGED_DIR)